# Tokenization - NLP

Tokenization is a process of converting text into tokens. We may need to convert or segment text into sentences, words, characters or subwords.

Machine does not understand text data as human. We need to convert it into tokens and then IDs.

## Word level tokenization

Word level tokenization involves splitting text into words

In [1]:
text = "Python is a programming language!"

In [2]:
# Approach 1

print(text.split())

['Python', 'is', 'a', 'programming', 'language!']


In [4]:
# Tokenization filtering out punctuations like !, ;, :, etc.
print([char for char in text.split() if char.isalnum()])

['Python', 'is', 'a', 'programming']


In [6]:
import re

print(re.split(r'(\s)', text))

['Python', ' ', 'is', ' ', 'a', ' ', 'programming', ' ', 'language!']


In [7]:
print(re.split(r'([.,:;?_!"()\']|--|\s)', text))

['Python', ' ', 'is', ' ', 'a', ' ', 'programming', ' ', 'language', '!', '']


In [8]:
print([char for char in re.split(r'([.,:;?_!"()\']|--|\s)', text) if char.strip()])

['Python', 'is', 'a', 'programming', 'language', '!']


## Character level tokenization

In [11]:
print([char for char in text if char.strip()])

['P', 'y', 't', 'h', 'o', 'n', 'i', 's', 'a', 'p', 'r', 'o', 'g', 'r', 'a', 'm', 'm', 'i', 'n', 'g', 'l', 'a', 'n', 'g', 'u', 'a', 'g', 'e', '!']


## Subword level tokenization

The following tokenization concepts are considered as subword tokenization:

- WordPiece
- Byte-pair encoding

## Unigram tokenization

## Sentence piece tokenization

## Word piece tokenization

WordPiece tokenizer is a sub word tokenization algorithm. It split words into smaller units until they are available in a fixed vocabulary.

WordPiece is a subword tokenization algorithm developed by Google, used in BERT.

**Core idea:** Split words into the smallest meaningful units (subwords) that exist in a fixed vocabulary, balancing between word-level and character-level tokenization.

---

### It solves two problems

| Problem | Naive approach | WordPiece fix |
|---|---|---|
| Unknown words (OOV) | `[UNK]` for anything unseen | Break into known subwords |
| Huge vocabulary | One token per word → millions | Shared subword pieces → compact vocab |

---

### Key properties

- Frequent words stay whole → `play` stays `play`
- Rare/unknown words get split → `playing` → `play` + `##ing`
- `##` prefix marks a **continuation** piece (not a word start)
- Worst case: falls back to characters → always some representation

---

### One-liner definition

> WordPiece tokenizer greedily splits each word into the longest possible subwords found in a fixed vocabulary, marking non-initial pieces with `##`, so that even unseen words get a meaningful representation.


In [16]:
def wordpiece_tokenize(words: list[str], vocab: set):
    """WordPiece tokenizer
    
    Takes words list as input and returns its tokenized list
    """
    all_tokens = []

    for word in words:
        start = 0
        tokens = []

        while start < len(word):
            end = len(word)
            found = None

            while start < end:
                substr = word[start:end]
                candidate = substr if start == 0 else "##" + substr

                if candidate in vocab:
                    found = candidate
                    break

                # Truncating from end
                end -= 1

            if found is None:
                tokens = ["[UNK]"]
                break

            tokens.append(found)
            start = end

        all_tokens.extend(tokens)
    
    return all_tokens


- playing  
      ^^^

- It removes each letter from end and check
- Then start will be end, the end was 4 when candidate found
- So, it will check for same word again but from start where it is 4

- ing  
  ^^^
- So, this time start is not 0, it will be ##ing
checks in vocab and flag as found.

In [17]:
vocab = {"play", "##ing", "##ed", "##er", "run", "##ning", "foot", "##ball"}

words = ["playing", "runner", "football"]
print(wordpiece_tokenize(words, vocab))
# ["play", "##ing", "run", "##ner"... wait "##ner" not in vocab → [UNK]]


['play', '##ing', '[UNK]', 'foot', '##ball']


### Encoder Tracing

Let's trace through tokenize(["playing"], vocab) where vocab = {"play", "##ing"}.

Outer loop — iterate over each word:


word = "playing"
start = 0
Inner loop — find longest matching prefix from start:


Iteration 1:
  start=0, end=7  →  substr = "playing"   candidate = "playing"    ❌ not in vocab
  start=0, end=6  →  substr = "playin"    candidate = "playin"     ❌ not in vocab
  start=0, end=5  →  substr = "playi"     candidate = "playi"      ❌ not in vocab
  start=0, end=4  →  substr = "play"      candidate = "play"       ✅ found!

  tokens = ["play"],  start = 4

Iteration 2:
  start=4, end=7  →  substr = "ing"       candidate = "##ing"      ✅ found!

  tokens = ["play", "##ing"],  start = 7

start=7 == len("playing")=7  →  loop ends
Result: ["play", "##ing"]

The two rules that drive it:

start == 0 → first chunk of word, no prefix → check substr as-is
start > 0 → continuation chunk → check "##" + substr
This is why ## naturally appears — it's not added manually, it's baked into how you look up the vocabulary after the first chunk.

What happens on no match ([UNK]):


word = "xyz",  vocab has no "x", "xy", "xyz"

  start=0, end shrinks all the way to 0
  found = None  →  tokens = ["[UNK]"],  break
The whole word collapses to [UNK] — even if later characters exist in vocab, once any chunk fails the entire word is unknown.

### WordPiece Decoding Tokenizer

In [18]:
lst = ["#h", "##g", "##h", "##f"]

def decode_wordpiece(tokens):
    words = []
    current = ""
    for token in tokens:
        if token.startswith("##"):
            current += token[2:]   # strip ## and append
        else:
            if current:
                words.append(current)
            current = token        # start new word
    if current:
        words.append(current)
    return words

print(decode_wordpiece(lst))
# ['#hghf']


['#hghf']


## Byte-pair encoding